# Python 기반 Front-end & Back-end 구현

#### 1) FastAPI 만으로 구현
#### 2) Gradio + FastAPI 으로 구현

## 1) FastAPI 만으로 구현


**get 기능만으로 최대한 간단하게 구현했습니다.**


In [ ]:
#pip install fastapi #웹 프로그램 만들기
#pip install uvicorn #FastAPI 서버를 실행하는 서버

from fastapi import FastAPI

#FastAPI 서버 만들기
app = FastAPI()

# 홈페이지 메인 제작
@app.get("/")
def home():
    return {"message": "매출채권회전율 계산기"}

#Get 요청이 오면 다음의 함수를 실행하라
@app.get("/ar_turnover") #홈페이지 만드는 코드
def ar_turnover(sales: float, ar: float):

    #평균 매출채권이 0인 경우
    if ar == 0:
        return {
            "error":"평균 매출채권이 0이므로 매출채권회전율을 계산할 수 없습니다."
        }

    turnover = sales / ar

    return{
        "순매출액": sales,
        "평균 매출채권": ar,
        "매출채권회전율": f"{turnover:.2f}회"
    }

```
웹 실행: python -m uvicorn ar_get_fastAPI:app --reload
```

```
웹 접속 공통: http://127.0.0.1:8000/

http://127.0.0.1:8000/docs → FastAPI가 자동으로 만들어주는 특별한 주소
http://127.0.0.1:8000/debt_ratio?equity=100&liability=200 → 실제 프로그램이 호출하는 API 결과
http://127.0.0.1:8000/ → 홈페이지 메인
```

## 2) Gradio + FastAPI 으로 구현

**실제 실행 시에는, VsCode 시스템으로 Gradio와 FastAPI 코드를 각각 py로 생성하여 실시하였습니다.**

**Gradio는 사용자 화면(UI) 구현, POST는 API 요청 처리를 위한 예제이므로 파일을 분리하였습니다.**

* GET은 URL의 쿼리 파라미터로 데이터를 전달하는 방식이고,
* POST는 요청 본문(Body)에 데이터를 담아 전송하는 방식입니다.
* 이번 예제에서는 매출액과 평균 매출채권을 하나의 JSON 객체로 받아 처리하는 API를 구현해 보고 싶어서 POST 방식을 사용했습니다.

In [ ]:
#ar_gradio_app.py에서 구현
from fastapi import FastAPI
from pydantic import BaseModel  # 사용자가 어떤 데이터를 보낼지 점검

app = FastAPI()

class FinancialData(BaseModel):
    sales: float
    ar: float

@app.post("/ar_turnover")
def calculate_ar_turnover(data: FinancialData):
    if data.ar == 0:
        return {"error": "평균 매출채권은 0이 될 수 없습니다."}

    turnover = data.sales / data.ar

    return {
        "ar_turnover": round(turnover, 2)
    }

In [ ]:
#ar_post_fastAPI에서 구현
#Gradio 화면 따로

import gradio as gr #화면 만들기
import requests #FastAPI와 통신하기 위한 목적

def calculate(sales, ar):
    #서버에 데이터를 보내는 POST 요청
    #답은 response에 저장. 숫자가 아니라 응답 데이터/상태코드/응답헤더/JSON내용
    response = requests.post(
        #뒤에서 작업할 주소
        "http://127.0.0.1:8000/ar_turnover",
        json={
            "sales": sales,
            "ar": ar
        }
    )

    result = response.json() #응답데이터를 JSON 형식으로 변환

    if "error" in result:
        return result["error"]

    return f"매출채권회전율은 {result['ar_turnover']:.2f}회입니다."

app = gr.Interface(
    fn=calculate,
    inputs=[
        gr.Number(label="순매출액"),
        gr.Number(label="평균 매출채권")
    ],

    outputs=gr.Textbox(label="계산 결과"),
    title="매출채권회전율 계산기"
)

app.launch()

* Gradio가 FastAPI API를 호출하는 구조이기 때문에,
* 먼저 FastAPI 서버를 실행해 요청을 받을 준비를 해야 합니다.
* 이후 Gradio를 실행하면 requests.post()를 통해 FastAPI와 정상적으로 통신할 수 있습니다.

```
웹 실행:
터미널 1) python -m uvicorn ar_post_fastapi:app --reload
터미널 2) python ar_gradio.py
```

```
웹 접속: http://127.0.0.1:7860
```